# 02 - Sensor Analysis (smartphone IMU + GNSS)

Investigate the sensor channels available for dead reckoning: accelerometer, gyroscope, magnetometer, gravity and orientation, plus GNSS quality. We use one synchronised trip (`Vw16b`).

In [ ]:
import sys, os
from pathlib import Path
for p in ("..", "."):
    if (Path(p) / "src").is_dir():
        sys.path.insert(0, os.path.abspath(p)); break

In [ ]:
import numpy as np, pandas as pd
from src.data.io_vnbd_loader import IOVNBDDataset
from src.data.smartphone_extractor import SmartphoneExtractor
ds = IOVNBDDataset("data/raw")
ex = SmartphoneExtractor(ds)
d = ex.extract_trip("vw16b").data
print("trip vw16b:", len(d), "samples")

## 1. IMU summary statistics

In [ ]:
imu = [c for c in d.columns if c.startswith(("accel", "gyro", "mag", "gravity"))]
d[imu].describe().T.round(4)

## 2. Acceleration magnitude over time

In [ ]:
import matplotlib.pyplot as plt
a = d[["accel_x", "accel_y", "accel_z"]].to_numpy()
mag = np.linalg.norm(a, axis=1)
t = d["timestamp"] - d["timestamp"].iloc[0]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, mag, lw=0.5)
ax.set(xlabel="time since start (s)", ylabel="|a| (m/s^2)", title="Acceleration magnitude")
plt.show()

## 3. Gyro channels

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
for ax, c in zip(axes, ["gyro_x", "gyro_y", "gyro_z"]):
    ax.plot(t, d[c], lw=0.5)
    ax.set_ylabel(c)
axes[-1].set_xlabel("time since start (s)")
plt.show()

## 4. Sensor distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, c in zip(axes, ["accel_x", "gyro_x", "mag_x"]):
    ax.hist(d[c].dropna(), bins=60, density=True)
    ax.set_title(c)
plt.tight_layout()
plt.show()

## 5. GNSS characteristics

In [ ]:
g = d.dropna(subset=["latitude_deg"])
print("GNSS fixes:", len(g))
print("speed_kmh stats:", g["speed_kmh"].describe().round(2).to_dict())
print("accuracy (m) stats:", g["position_accuracy_m"].describe().round(2).to_dict())
print("sats stats:", g["gps_satellites"].describe().round(1).to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
tg = g["timestamp"] - d["timestamp"].iloc[0]
ax.plot(tg, g["speed_kmh"], lw=0.8)
ax.set(xlabel="time since start (s)", ylabel="speed (kmh)", title="GNSS speed profile")
plt.show()

## 6. Takeaways

- Acceleration magnitude hovers around gravitational constant (9.8) plus dynamics, so gravity dominates raw accel; gravity channels are available separately.
- Gyro rates are small (rad/s) as expected for road driving.
- GNSS fixes are bursty; the pipeline forward-fills them onto the 10 Hz IMU grid.